![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Loading

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

### What this notebook is for

Turning the raw file you explored yesterday into a relational database you can query. This is the engineering day: clean, normalise, validate, load.

The order matters and it is not negotiable. Lookup tables are built and loaded **before** the fact table that references them, because a foreign key pointing at a row that does not exist yet is not a foreign key.

> **Done when:** `sql/schema.sql` is written, `data/clean/` holds one CSV per table, the `.db` file exists with the right row count in every table, `check_fk` passes on every relationship, and a test join returns rows.

### The one thing that goes wrong

**Dangling foreign keys.** SQLite will accept them unless you tell it not to, your joins will silently drop rows, and you will find out on Wednesday when a number looks wrong. Section 6 is the check that prevents it. Do not skip it.

---
## 0. Setup

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

DB = "../data/project.db"      # gitignored: a database is a build artefact

In [ ]:
df = load_raw("cash_requests.csv")     # your dataset
df.shape

---
## 1. Fix the types

Everything arrives from CSV as text or float. Three fixes, in this order:

1. **Dates** — `pd.to_datetime`. Use `errors="coerce"` so unparseable values become `NaT` instead of raising, then count how many you lost.
2. **Numbers stored as text** — `"$120.00"`, `"1,234"`, `"45%"`. Strip the symbols with `.str.replace`, then `pd.to_numeric`.
3. **Booleans** — `"t"`/`"f"` and `"Yes"`/`"No"` become real booleans, or 0/1 integers, which is what SQLite stores anyway.

Get this right now. A date column left as text sorts `"10/01"` before `"9/01"`, and your Wednesday trend line will be nonsense.

In [ ]:
# Dates
date_cols = ["created_at", "updated_at"]        # yours
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="mixed")

# How many failed to parse? Compare against the null count you saw yesterday.
df[date_cols].isna().sum()

In [ ]:
# Numbers hiding in strings. Airbnb's `price` is the classic: "$120.00"
# df["price"] = (df["price"].astype(str)
#                           .str.replace(r"[$,]", "", regex=True)
#                           .pipe(pd.to_numeric, errors="coerce"))

# Booleans
# df["host_is_superhost"] = df["host_is_superhost"].map({"t": 1, "f": 0})

df.dtypes.value_counts()

---
## 2. Handle nulls and duplicates

Work from the decision table you filled in notebook 01, section 3. Each column gets one of four treatments:

- **Drop the column** — mostly empty, or you do not need it.
- **Drop the rows** — only defensible when few rows are affected. Print how many you lose.
- **Fill** — with a sensible constant, a median, or `"Unknown"`. Say why in a comment.
- **Leave it** — a null that means something (never repaid, no review yet) is data. Keep it and let SQL's `IS NULL` do the work.

Whatever you choose, **print the row count before and after**. A silent `dropna()` that removes 60% of your data is the mistake nobody notices.

In [ ]:
before = len(df)

# Drop the columns you will not use
# df = df.drop(columns=["col_a", "col_b"])

# Drop rows only where it is defensible - e.g. no primary key
# df = df.dropna(subset=["id"])

# Fill where a default makes sense
# df["recovery_status"] = df["recovery_status"].fillna("none")

print(f"{before:,} -> {len(df):,} rows  ({before - len(df):,} dropped)")

In [ ]:
# Exact duplicates
df = df.drop_duplicates()

# And confirm the primary key is now unique
duplicate_report(df, subset=["id"])

---
## 3. Standardise the categories

Before a column can become a lookup table, its values have to be consistent. `"Madrid"`, `"madrid"` and `" Madrid"` are three rows in your lookup table and three broken joins.

Strip whitespace, settle on one case, and map any variants onto one spelling. Then re-run `value_counts()` and check the distinct count is what you expect from notebook 01.

In [ ]:
cat_cols = ["status", "transfer_type"]      # yours

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Fold any remaining variants onto one spelling
# df["status"] = df["status"].replace({"cancelled": "canceled"})

for col in cat_cols:
    print(f"{col}: {df[col].nunique()} distinct -> {sorted(df[col].unique())[:8]}")

In [ ]:
# Blank strings became the literal "nan" or "" above. Turn them back into nulls.
df = df.replace({"": None, "nan": None, "none": None})

duplicate_report(df)["blank_strings"]

---
## 4. Design your tables and draw the ERD

Stop coding for ten minutes. Fill in the table below first, then draw it.

**The ERD is a deliverable.** Draw it in [Excalidraw](https://excalidraw.com) or [draw.io](https://app.diagrams.net), export it as a PNG, commit it, and link it from your README. It is also slide 4 of your presentation, where you defend these decisions.

### The design

| Table | Type | Primary key | Foreign keys | Columns | Approx. rows |
|---|---|---|---|---|---|
|  | lookup |  | — |  |  |
|  | lookup |  | — |  |  |
|  | dimension |  | — |  |  |
|  | **fact** |  |  |  |  |

### The relationships

| From | To | Cardinality | Read it as |
|---|---|---|---|
|  |  | N:1 | "many … belong to one …" |

### ERD

Commit the image and embed it here:

`![ERD](../erd.png)`

Then write `sql/schema.sql` to match. The two must agree — a schema that does not match the diagram is the thing a reviewer notices first.

---
## 5. Build the tables

Now execute the design. Lookups first, then the dimension, then the fact table.

`make_lookup` and `add_foreign_key` are **`TODO`s in `src/functions.py`** — their docstrings lay out the decisions (drop nulls or keep an "Unknown" row, sort before assigning ids or not, start at 0 or 1) and you make them. Write them once and the rest of this section is three lines each.

In [ ]:
# One lookup table per categorical column you chose.
status_lookup = make_lookup(df, "status")
status_lookup

In [ ]:
transfer_type_lookup = make_lookup(df, "transfer_type")
transfer_type_lookup

In [ ]:
# A dimension is a lookup with attributes: aggregate per entity rather than
# just listing distinct labels.
# users = (df.groupby("user_id")
#            .agg(first_request=("created_at", "min"),
#                 n_requests=("id", "count"))
#            .reset_index())
# users.head()

In [ ]:
# The fact table: swap each text column for its foreign key.
fact = df.copy()
fact = add_foreign_key(fact, status_lookup, "status")
fact = add_foreign_key(fact, transfer_type_lookup, "transfer_type")

fact.head()

In [ ]:
# Keep only the columns your schema declares. A fact table carrying 60 unused
# columns is not a design, it is the raw file with extra steps.
fact = fact[[
    "id",
    # your measures
    # your foreign keys
]]
fact.shape

---
## 6. Validate the foreign keys — **do not skip this**

`check_fk` checks that every foreign-key value in the child table exists in the parent. It must return `ok: True` for every relationship before you load anything.

If it reports orphans, one of three things happened:

- **A cleaning step ran after the lookup was built**, so the values no longer match. Rebuild the lookup from the cleaned column.
- **You used an inner merge in `add_foreign_key`** and it dropped rows — check your row count against section 2.
- **The parent genuinely does not contain those keys.** This is common with two real files: Airbnb's `reviews.csv` references listings that are no longer in the snapshot. That is a legitimate finding, and the fix is a decision you document: drop the orphaned children, or keep them with a null key.

`n_null` is reported separately, because a nullable foreign key can be a deliberate choice. Orphans never are.

In [ ]:
checks = {
    "fact.status_id -> status_lookup":               check_fk(fact, status_lookup, "status_id"),
    "fact.transfer_type_id -> transfer_type_lookup": check_fk(fact, transfer_type_lookup, "transfer_type_id"),
}

for name, result in checks.items():
    flag = "PASS" if result["ok"] else "FAIL"
    print(f"{flag}  {name}: {result['n_orphans']:,} orphans, {result['n_null']:,} nulls")
    if not result["ok"]:
        print(f"      examples: {result['orphan_examples']}")

In [ ]:
assert all(r["ok"] for r in checks.values()), "Fix the orphaned keys before loading"
print("All foreign keys valid.")

---
## 7. Export one CSV per table

`data/clean/` is your handover from Python to SQL, and it is gitignored — the notebook rebuilds it, so there is no reason to commit it.

Build the dict **in load order**: lookups and dimensions first, fact table last. Python dicts keep insertion order, and `load_to_sql` relies on it.

In [ ]:
tables = {
    # load order: no foreign keys first
    "status_lookup":        status_lookup,
    "transfer_type_lookup": transfer_type_lookup,
    # "users":              users,
    "cash_requests":        fact,        # the fact table goes last
}

export_clean(tables)

---
## 8. Create the database and load it

Two steps in one call. `load_to_sql` runs your `schema.sql` to create the tables with your declared types and foreign-key constraints, then `to_sql`s each DataFrame into them in dict order.

It also sets `PRAGMA foreign_keys = ON`, so a bad key now raises instead of slipping through — which is why section 6 comes first.

> **Watch `replace`.** `replace=True` drops each table and lets pandas invent the schema, throwing away your types and constraints. Useful while iterating; do not submit with it on.

In [ ]:
import os
if os.path.exists(DB):
    os.remove(DB)          # start clean, so re-running this notebook is repeatable

load_to_sql(tables, DB, schema_file="../sql/schema.sql")

---
## 9. Verify

Two checks. Row counts prove the data arrived; a test join proves the keys line up. Both have to pass before you touch notebook 03 — debugging a join tomorrow is much more expensive than catching it now.

In [ ]:
table_counts(DB)

Compare against section 7. A table showing 0 rows loaded but inserted nothing, which almost always means it went in before its parent.

In [ ]:
# A test join. Every lookup-table label should come back, with sensible counts.
run_query('''
    SELECT s.status,
           COUNT(*) AS n_requests
    FROM cash_requests c
    JOIN status_lookup s ON c.status_id = s.status_id
    GROUP BY s.status
    ORDER BY n_requests DESC
''', DB)

In [ ]:
# And the tables the database thinks it has, which is what a reviewer will run.
run_query("SELECT name FROM sqlite_master WHERE type='table'", DB)

### Where you are

- [ ] Types fixed, nulls decided, categories standardised
- [ ] ERD drawn, committed, and matching `sql/schema.sql`
- [ ] `check_fk` passes on every relationship
- [ ] One CSV per table in `data/clean/`
- [ ] `.db` built, row counts correct, test join returns rows
- [ ] Committed and pushed

Open the `.db` file in **DB Browser for SQLite** and click through it. Seeing your own schema in a database browser is the moment this stops being abstract.

Next: **[03_hypothesis_and_visualization.ipynb](03_hypothesis_and_visualization.ipynb)** — query it, chart it, report it.